# 02 · Registrar uma coleta nova e selecionar 200 patentes
**Piloto Qwen3-8B · CPU · execução após a auditoria 00.** Este notebook liga a Etapa 1 à coleta. Com uma exportação autorizada, verifica a origem declarada, seleciona uma patente por família e congela as partições. O diagnóstico de GPU pode ser feito em paralelo com a obtenção da exportação.

1. No [AI Pat+](https://aipat.cnki.net/), execute uma **nova consulta** de pedidos e concessões de invenção na China, em chinês. Refaça o recorte “Life Sciences” nos campos de palavras-chave, título DWPI ou resumo DWPI com a fusão de famílias da § 4.1. Registre a expressão exatamente como foi aceita pela plataforma e as opções de família. Use datas de **depósito de 01/01/2000 a 30/06/2025**. A amostra usa as três janelas de 2010 a 2024. Exporte candidatos suficientes em cada janela. Se houver limite da plataforma, registre a paginação, ordenação e cobertura da exportação.
2. Salve a exportação nova em CSV UTF-8 em `dados/brutos/piloto_200/exportacao.csv`. Se for preciso converter uma planilha, preserve o original e registre a conversão nas observações. Mantenha números de publicação e família como texto. Envie dados ao Drive apenas se a licença permitir.
3. O seletor reconhece os cabeçalhos `id_publicacao, id_familia, data_deposito, data_publicacao, titulo_zh, resumo_zh, cpc_principal`. Prefira datas ISO `AAAA-MM-DD`. Para outros cabeçalhos, indique a correspondência na célula de parâmetros. A coluna CPC pode ficar ausente; sua cobertura condiciona o futuro treino supervisionado.
4. Execute as células. Sem o CSV ou a declaração da coleta, o notebook informa a pendência. Ele não preenche patentes nem usa material de projetos anteriores.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib
import json
import subprocess
import sys
from datetime import date, datetime, timezone
from zoneinfo import ZoneInfo

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/Replicação LLM EIDT IC"  # @param {type:"string"}
raiz = Path(PASTA_DRIVE).resolve(strict=True)
montagem = Path("/content/drive").resolve(strict=True)
if raiz == montagem or not raiz.is_relative_to(montagem):
    raise ValueError("Selecione a pasta do projeto no Drive.")
brutos = raiz / "dados/brutos/piloto_200"
entrada = brutos / "exportacao.csv"
recibo = brutos / "proveniencia.json"
destino = raiz / "dados/processados/piloto_200"
resultados = raiz / "resultados/etapa_1/piloto_200"
seletor = raiz / "scripts/selecionar_piloto.py"
configuracao = raiz / "config/experimento.json"

def hash_arquivo(caminho):
    with caminho.open("rb") as arquivo:
        return hashlib.file_digest(arquivo, "sha256").hexdigest()

indice = {}
for linha in (raiz / "SHA256SUMS.txt").read_text(encoding="utf-8-sig").splitlines():
    assinatura, nome = linha.split("  ", 1)
    indice[nome] = assinatura
for arquivo in (seletor, configuracao):
    if hash_arquivo(arquivo) != indice.get(arquivo.relative_to(raiz).as_posix()):
        raise ValueError(f"Recopie {arquivo.name} e o índice da entrega atual do Desktop.")
brutos.mkdir(parents=True, exist_ok=True)
print("Exportação esperada:", entrada)

## 1. Declarar a origem
Na primeira execução com dados, preencha a data real da nova exportação, a consulta e as opções aplicadas. Inclua em **cobertura** se exportou todos os resultados ou um subconjunto e como o escolheu. Marque a confirmação somente depois de conferir os filtros, a licença e a procedência. O hash vincula o registro a um CSV específico. Ao repetir, um recibo existente é lido e validado. Não inclua credenciais.

In [ ]:
DATA_EXPORTACAO = ""  # @param {type:"string"}
CONSULTA_EXECUTADA = ""  # @param {type:"string"}
OPCOES_FAMILIA = ""  # @param {type:"string"}
COBERTURA_E_CONVERSAO = ""  # @param {type:"string"}
CONFIRMO_NOVA_EXPORTACAO_E_FILTROS = False  # @param {type:"boolean"}
# Preencha somente os nomes que divergem dos canônicos. Exemplo:
# MAPA = {"id_publicacao": "Publication Number", "id_familia": "Family ID"}
MAPA = {}

def validar_recibo(dados, hash_atual):
    campos = ("data_exportacao", "consulta", "opcoes_familia", "cobertura_e_conversao")
    if any(not isinstance(dados.get(c), str) or not dados[c].strip() for c in campos):
        raise ValueError("Informe data, consulta, regra de família e cobertura da exportação.")
    dia = date.fromisoformat(dados["data_exportacao"])
    hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
    if not date(2026, 11, 2) <= dia <= hoje:
        raise ValueError("Informe a data real desta nova exportação (desde 02/11/2026, sem data futura).")
    if (dados.get("fonte") != "AI Pat+" or dados.get("confirmacao_operador") is not True
            or dados.get("sha256_exportacao") != hash_atual):
        raise ValueError("O recibo não corresponde à coleta declarada ou aos bytes deste CSV.")

origem = None
if not entrada.is_file():
    print("Pendente: coloque a nova exportação no caminho acima. Nenhuma amostra foi selecionada.")
elif entrada.stat().st_size == 0:
    raise ValueError("A exportação está vazia.")
else:
    hash_entrada = hash_arquivo(entrada)
    if recibo.is_file():
        existente = json.loads(recibo.read_text(encoding="utf-8-sig"))
        validar_recibo(existente, hash_entrada)
        origem = existente
    elif CONFIRMO_NOVA_EXPORTACAO_E_FILTROS:
        novo = {
            "fonte": "AI Pat+", "data_exportacao": DATA_EXPORTACAO.strip(),
            "consulta": CONSULTA_EXECUTADA.strip(), "opcoes_familia": OPCOES_FAMILIA.strip(),
            "cobertura_e_conversao": COBERTURA_E_CONVERSAO.strip(),
            "filtros_confirmados": ["China", "chinês", "pedido ou concessão de invenção",
                                    "depósito 2000-01-01 a 2025-06-30", "ciências da vida"],
            "confirmacao_operador": True, "sha256_exportacao": hash_entrada,
            "registro_utc": datetime.now(timezone.utc).isoformat(),
        }
        validar_recibo(novo, hash_entrada)
        with recibo.open("x", encoding="utf-8") as f:
            json.dump(novo, f, ensure_ascii=False, indent=2)
            f.write("\n")
        origem = novo
    else:
        print("CSV presente. Preencha e confirme sua declaração para registrar a origem.")
if origem:
    print("Recibo conferido:", recibo.name, "| exportação SHA-256:", origem["sha256_exportacao"])

## 2. Selecionar e verificar
O seletor ordena famílias com uma regra determinística e a semente 42. As cotas cronológicas são 67, 67 e 66. No conjunto completo, são 160 famílias para treino, 20 para validação e 20 para teste. A falta de famílias em qualquer janela interrompe a seleção. Consulte o registro de exclusões e a cobertura de CPC no relatório. Ao mudar a origem ou o mapeamento depois de congelar a amostra, preserve a execução anterior em um projeto versionado.

In [ ]:
if origem is None:
    print("Seleção pendente de exportação e recibo.")
else:
    resultados.mkdir(parents=True, exist_ok=True)
    comando = [sys.executable, str(seletor), "--entrada", str(entrada), "--saida", str(destino),
               "--config", str(configuracao)]
    if MAPA:
        texto_mapa = json.dumps(MAPA, ensure_ascii=False, sort_keys=True, indent=2)
        chave_mapa = hashlib.sha256(texto_mapa.encode()).hexdigest()[:16]
        arquivo_mapa = resultados / f"mapa_colunas_{chave_mapa}.json"
        if not arquivo_mapa.exists():
            arquivo_mapa.write_text(texto_mapa, encoding="utf-8")
        if arquivo_mapa.read_text(encoding="utf-8") != texto_mapa:
            raise ValueError("O mapa salvo diverge dos parâmetros da célula.")
        comando.extend(["--mapa", str(arquivo_mapa)])
    subprocess.run(comando, check=True)
    auditoria = json.loads((destino / "auditoria.json").read_text(encoding="utf-8"))
    if (origem["sha256_exportacao"] != auditoria["sha256_entrada"]
            or hash_arquivo(destino / "piloto_200.csv") != auditoria["sha256_piloto"]):
        raise ValueError("O recibo, a auditoria e a amostra apontam para conteúdos diferentes.")
    ligacao = {"sha256_recibo": hash_arquivo(recibo), "sha256_exportacao": auditoria["sha256_entrada"],
               "sha256_piloto": auditoria["sha256_piloto"], "sha256_auditoria": hash_arquivo(destino / "auditoria.json"),
               "sha256_config": hash_arquivo(configuracao), "sha256_seletor": hash_arquivo(seletor)}
    texto = json.dumps(ligacao, ensure_ascii=False, sort_keys=True, indent=2) + "\n"
    comprovante = resultados / f"conferencia_{hashlib.sha256(texto.encode()).hexdigest()[:16]}.json"
    if not comprovante.exists():
        comprovante.write_text(texto, encoding="utf-8")
    if comprovante.read_text(encoding="utf-8") != texto:
        raise ValueError("O comprovante existente diverge da execução atual.")
    print("Amostra:", destino / "piloto_200.csv")
    for chave in ("particoes", "janelas", "status_cpc"):
        print(chave, auditoria[chave])
    print("Comprovante:", comprovante)

## 3. Antes de seguir para o treino
Abra a auditoria e confira a cobertura por janela e por partição. Registre os materiais em `config/insumos_piloto_200.csv` e rode o notebook 00 de novo se quiser atualizar o inventário. Reserve o teste até congelar configuração e prompts. Antes de interpretar o F1 de extração, anote uma referência humana independente para os seis campos. O CPC supervisiona a adaptação de domínio. Para medir EDh histórico e sua concordância com Dh, será necessário ampliar a anterioridade observada e obter o grafo de citações.